# DATA ANALYTICS ML CHALLENGE / 2026-2
## Laboratorio 1. Modelos supervisados y métricas (Semana 12)

Profesor: Elias Buitrago Bolivar  
Universidad Central  
Caso: Brazilian E-Commerce Public Dataset by Olist

Objetivo: construir un modelo base para anticipar entregas tardías a partir de datos de compras, productos, clientes, vendedores y pagos.

Las instrucciones de GitHub y entrega se encuentran en el documento técnico. Las secciones de carga y preparación se suministran resueltas; complete las cuatro actividades de modelado y evaluación.

## 1. Carga de datos

Una observación del modelo corresponde a un pedido, aunque tenga varios productos y pagos. Se integrarán siete tablas del conjunto público de Olist.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error)

In [2]:
ORIGEN = ('https://raw.githubusercontent.com/spdrio/'
          'Brazilian-E-Commerce-Public-Dataset-by-Olist/'
          'fa0cc6d1aff713d3c8b9a1e633b56889881848bf/files')

pedidos = pd.read_csv(f'{ORIGEN}/olist_orders_dataset.csv')
items = pd.read_csv(f'{ORIGEN}/olist_order_items_dataset.csv')
productos = pd.read_csv(f'{ORIGEN}/olist_products_dataset.csv')
clientes = pd.read_csv(f'{ORIGEN}/olist_customers_dataset.csv')
vendedores = pd.read_csv(f'{ORIGEN}/olist_sellers_dataset.csv')
pagos = pd.read_csv(f'{ORIGEN}/olist_order_payments_dataset.csv')
categorias = pd.read_csv(f'{ORIGEN}/product_category_name_translation.csv')

print('Pedidos:', pedidos.shape)
print('Items:', items.shape)
print('Productos:', productos.shape)
print('Clientes:', clientes.shape)
print('Vendedores:', vendedores.shape)
print('Pagos:', pagos.shape)

Pedidos: (99441, 8)
Items: (112650, 7)
Productos: (32951, 9)
Clientes: (99441, 5)
Vendedores: (3095, 4)
Pagos: (103886, 5)


## 2. Preparación de variables

Un pedido puede contener varios ítems y pagos. Primero se resumen estas tablas por `order_id`, para evitar contar un mismo pedido varias veces. Las variables de destino se construyen después y no se incluyen entre los predictores.

In [3]:
# Información de los productos y vendedores incluidos en cada pedido.
items = items.merge(productos, on='product_id', how='left', validate='many_to_one')
items = items.merge(vendedores, on='seller_id', how='left', validate='many_to_one')
items = items.merge(categorias, on='product_category_name',
                    how='left', validate='many_to_one')

items['volumen_cm3'] = (items['product_length_cm'] * items['product_height_cm']
                       * items['product_width_cm'])
items['categoria'] = items['product_category_name_english'].fillna(
    items['product_category_name']).fillna('sin_categoria')

# Resumen de productos: una fila por pedido.
resumen_items = items.groupby('order_id', as_index=False).agg(
    valor_productos=('price', 'sum'),
    valor_flete=('freight_value', 'sum'),
    precio_maximo=('price', 'max'),
    cantidad_items=('order_item_id', 'count'),
    cantidad_productos=('product_id', 'nunique'),
    cantidad_vendedores=('seller_id', 'nunique'),
    peso_total_g=('product_weight_g', 'sum'),
    peso_maximo_g=('product_weight_g', 'max'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    descripcion_promedio=('product_description_lenght', 'mean'),
    nombre_promedio=('product_name_lenght', 'mean'),
)

# Categoría y vendedor del producto de mayor precio del pedido.
principal = (items.sort_values(['order_id', 'price', 'order_item_id'],
                                ascending=[True, False, True])
             .drop_duplicates('order_id'))
principal = principal[['order_id', 'categoria', 'seller_state', 'seller_city']]
principal = principal.rename(columns={
    'categoria': 'categoria_principal',
    'seller_state': 'estado_vendedor',
    'seller_city': 'ciudad_vendedor'
})

In [4]:
# Los pagos también pueden aparecer en varias filas por pedido.
resumen_pagos = pagos.groupby('order_id', as_index=False).agg(
    numero_pagos=('payment_sequential', 'count'),
    cuotas_maximas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
)

pago_principal = (pagos.sort_values(['order_id', 'payment_value'],
                                    ascending=[True, False])
                  .drop_duplicates('order_id'))
pago_principal = pago_principal[['order_id', 'payment_type']].rename(
    columns={'payment_type': 'tipo_pago_principal'})

# Integración de las tablas: una fila por pedido.
data = pedidos.merge(clientes[['customer_id', 'customer_state', 'customer_city']],
                     on='customer_id', how='inner', validate='one_to_one')
data = data.merge(resumen_items, on='order_id', how='inner', validate='one_to_one')
data = data.merge(principal, on='order_id', how='left', validate='one_to_one')
data = data.merge(resumen_pagos, on='order_id', how='left', validate='one_to_one')
data = data.merge(pago_principal, on='order_id', how='left', validate='one_to_one')

assert data['order_id'].is_unique, 'Hay pedidos duplicados después de las uniones.'
print('Pedidos integrados:', len(data))

Pedidos integrados: 98666


In [5]:
# Se consideran pedidos entregados con fechas completas y coherentes.
fechas = ['order_purchase_timestamp', 'order_estimated_delivery_date',
          'order_delivered_customer_date']
for columna in fechas:
    data[columna] = pd.to_datetime(data[columna], errors='coerce')

data = data.loc[data['order_status'] == 'delivered'].dropna(subset=fechas).copy()
compra = data['order_purchase_timestamp'].dt.normalize()
prevista = data['order_estimated_delivery_date'].dt.normalize()
real = data['order_delivered_customer_date'].dt.normalize()

data['dias_prometidos'] = (prevista - compra).dt.days
data['dias_entrega'] = (real - compra).dt.days
data['is_late'] = (real > prevista).astype(int)
data['mes_compra'] = compra.dt.month
data['dia_semana_compra'] = compra.dt.dayofweek
data['dia_mes_compra'] = compra.dt.day
data['hora_compra'] = data['order_purchase_timestamp'].dt.hour
data['compra_fin_semana'] = (data['dia_semana_compra'] >= 5).astype(int)

data['vendedor_mismo_estado'] = (
    data['customer_state'] == data['estado_vendedor']).astype(int)
data['vendedor_misma_ciudad'] = (
    data['vendedor_mismo_estado'].eq(1) &
    data['customer_city'].str.lower().eq(data['ciudad_vendedor'].str.lower())
).astype(int)
data['proporcion_flete'] = data['valor_flete'] / (
    data['valor_productos'] + data['valor_flete']).replace(0, np.nan)

data = data.loc[(data['dias_prometidos'] >= 0) &
                (data['dias_entrega'] >= 0)].copy()
data = data.sort_values('order_id').reset_index(drop=True)

print('Pedidos disponibles:', len(data))
print('Entregas tardías:', round(100 * data['is_late'].mean(), 2), '%')

Pedidos disponibles: 96470
Entregas tardías: 6.77 %


### Variables del modelo

Se utilizarán datos que razonablemente están disponibles al cierre de la compra: plazo prometido, calendario, contenido del pedido, dimensiones de los productos, ubicación y medio de pago. Las fechas reales de envío y entrega, y las reseñas posteriores, quedan excluidas de las variables predictoras.

La lista de variables y sus tablas de origen se encuentra en el diccionario técnico anexo.

In [6]:
columnas_numericas = [
    'dias_prometidos', 'mes_compra', 'dia_semana_compra',
    'dia_mes_compra', 'hora_compra', 'compra_fin_semana',
    'valor_productos', 'valor_flete', 'precio_maximo',
    'cantidad_items', 'cantidad_productos', 'cantidad_vendedores',
    'peso_total_g', 'peso_maximo_g', 'volumen_total_cm3',
    'fotos_promedio', 'descripcion_promedio', 'nombre_promedio',
    'numero_pagos', 'cuotas_maximas', 'valor_pagado',
    'vendedor_mismo_estado', 'vendedor_misma_ciudad', 'proporcion_flete'
]
columnas_categoricas = [
    'customer_state', 'estado_vendedor',
    'categoria_principal', 'tipo_pago_principal'
]
columnas = columnas_numericas + columnas_categoricas

assert data['order_id'].is_unique
assert len(data) >= 35000
print('Variables predictoras:', len(columnas))
print('Numéricas:', len(columnas_numericas), '| Categóricas:', len(columnas_categoricas))

Variables predictoras: 28
Numéricas: 24 | Categóricas: 4


### Muestra individual

Cada estudiante utilizará 35.000 pedidos, seleccionados a partir del ID numérico entregado por el profesor (por ejemplo, ID 01). El mismo ID genera siempre la misma muestra y las mismas particiones.

Se utilizarán 24.500 pedidos para entrenar, 5.250 para validar y 5.250 para la prueba final. Durante las semanas 12 a 15 no se utilizarán los datos de prueba para elegir modelos.

En la siguiente celda cambie `ID_ESTUDIANTE` por el número asignado por el profesor. Por ejemplo, el ID 01 se escribe como `1`.


In [7]:
# Escriba el ID que le asignó el profesor (01 se escribe 1).
ID_ESTUDIANTE = 1
assert isinstance(ID_ESTUDIANTE, int) and 1 <= ID_ESTUDIANTE <= 30, (
    'El ID debe ser un número entero entre 1 y 30.')

semilla = 1000 + ID_ESTUDIANTE
identificador = f'{ID_ESTUDIANTE:02d}'

muestra, _ = train_test_split(data, train_size=35000,
                             random_state=semilla, stratify=data['is_late'])
muestra = muestra.sort_values('order_id').reset_index(drop=True)

train, temporal = train_test_split(muestra, test_size=0.30,
                                  random_state=semilla, stratify=muestra['is_late'])
valid, test = train_test_split(temporal, test_size=0.50,
                               random_state=semilla, stratify=temporal['is_late'])

X_train, y_train = train[columnas], train['is_late']
X_valid, y_valid = valid[columnas], valid['is_late']

huella_muestra = hashlib.sha256('|'.join(muestra['order_id']).encode()).hexdigest()[:12]
print('Estudiante ID:', identificador, '| Semilla:', semilla)
print('Muestra:', len(muestra), '| Train:', len(train),
      '| Valid:', len(valid), '| Test:', len(test))
print('Huella de muestra:', huella_muestra)

Estudiante ID: 01 | Semilla: 1001
Muestra: 35000 | Train: 24500 | Valid: 5250 | Test: 5250
Huella de muestra: 9e0cf0664d9d


## 3. Modelo de referencia y clasificación

Las variables numéricas se imputan y escalan; las categóricas se transforman con *One-Hot Encoding*. Estas transformaciones se ajustan únicamente con los datos de entrenamiento, dentro del `Pipeline`.

In [8]:
preprocesamiento = ColumnTransformer([
    ('numericas', make_pipeline(SimpleImputer(strategy='median'),
                               StandardScaler()), columnas_numericas),
    ('categoricas', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore')),
     columnas_categoricas)
])

referencia = DummyClassifier(strategy='most_frequent')
referencia.fit(X_train, y_train)
pred_ref = referencia.predict(X_valid)

print('Accuracy referencia:', round(accuracy_score(y_valid, pred_ref), 3))
print('F1 referencia:', round(f1_score(y_valid, pred_ref, zero_division=0), 3))

Accuracy referencia: 0.932
F1 referencia: 0.0


### Actividad 1. Entrenamiento

Complete el entrenamiento de la regresión logística usando `preprocesamiento`. Pruebe `class_weight=None` o `'balanced'`, y explique su elección en la sección de interpretación. Conserve las variables `modelo` y `pred`.

En el estimador utilice `class_weight=peso_clases` y `max_iter=1000` como límite de iteraciones. Si aparece una advertencia de convergencia, revise el entrenamiento.


In [9]:
# TODO 1. Entrene el modelo y genere predicciones.
# Decisión sobre class_weight: se compara en validación (ver celda siguiente) y se elige aquí.
peso_clases = 'balanced'

modelo = make_pipeline(
    preprocesamiento,
    LogisticRegression(class_weight=peso_clases, max_iter=1000)
)
modelo.fit(X_train, y_train)      # el preprocesamiento y el modelo se ajustan SOLO con entrenamiento
pred = modelo.predict(X_valid)    # predicciones 0/1 sobre validación (umbral 0,5)

print('Modelo entrenado con class_weight =', peso_clases)
print('Predicciones de validación:', len(pred), '| predichos como tardíos:', int(pred.sum()))

# Comparación auxiliar para justificar class_weight (no modifica `modelo` ni `pred`).
from sklearn.base import clone
comparacion = []
for cw in [None, 'balanced']:
    m = make_pipeline(clone(preprocesamiento), LogisticRegression(class_weight=cw, max_iter=1000))
    m.fit(X_train, y_train)
    p, pr = m.predict(X_valid), m.predict_proba(X_valid)[:, 1]
    comparacion.append({'class_weight': str(cw),
                        'accuracy': accuracy_score(y_valid, p),
                        'precision': precision_score(y_valid, p, zero_division=0),
                        'recall': recall_score(y_valid, p, zero_division=0),
                        'f1': f1_score(y_valid, p, zero_division=0),
                        'roc_auc': roc_auc_score(y_valid, pr),
                        'ap': average_precision_score(y_valid, pr),
                        'alertas': int(p.sum()),
                        'costo_R$': 8 * int(((p == 1) & (y_valid == 0)).sum()) + 35 * int(((p == 0) & (y_valid == 1)).sum()),
                        'iteraciones': int(m[-1].n_iter_[0])})
display(pd.DataFrame(comparacion).round(4))
# Las iteraciones usadas (< 1000) confirman que no hubo problema de convergencia.
# La advertencia 'OptimizeWarning: Unknown solver options: iprint' es de compatibilidad scipy/sklearn, no de convergencia.

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Modelo entrenado con class_weight = balanced
Predicciones de validación: 5250 | predichos como tardíos: 1827


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


,class_weight,accuracy,precision,recall,f1,roc_auc,ap,alertas,costo_R$,iteraciones
0,None,0.9326,1.000,0.0028,0.0056,0.7126,0.1604,1,12390,67
1,balanced,0.6686,0.121,0.6225,0.2026,0.7074,0.1600,1827,17538,141


## 4. Evaluación

### Actividad 2. Métricas

Calcule Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (AP) en validación. Para las métricas AUC use las probabilidades que estima el modelo.

In [10]:
# TODO 2. Calcule las métricas y muestre la matriz de confusión.
# Probabilidad estimada de "tardío" (clase 1): las métricas AUC dependen del orden de las probabilidades.
proba = modelo.predict_proba(X_valid)[:, 1]
medidas = {
    'accuracy': accuracy_score(y_valid, pred),
    'precision': precision_score(y_valid, pred, zero_division=0),
    'recall': recall_score(y_valid, pred, zero_division=0),
    'f1': f1_score(y_valid, pred, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, proba),
    'ap': average_precision_score(y_valid, proba),
}

# Filas = real, columnas = predicho -> [[TN, FP], [FN, TP]]
matriz = confusion_matrix(y_valid, pred, labels=[0, 1])
print('Matriz de confusión [[TN, FP], [FN, TP]]:')
print(matriz)
print('\nPrevalencia de tardíos en validación:', round(y_valid.mean(), 4))
display(pd.DataFrame([medidas]).round(4))

Matriz de confusión [[TN, FP], [FN, TP]]:
[[3289 1606]
 [ 134  221]]

Prevalencia de tardíos en validación: 0.0676


,accuracy,precision,recall,f1,roc_auc,ap
0,0.6686,0.121,0.6225,0.2026,0.7074,0.16


### Actividad 3. Errores por segmento

Compare la tasa de error por mes de compra y estado del cliente. Muestre tanto el porcentaje de error como la cantidad de pedidos de cada segmento.

In [11]:
assert pred is not None, 'Complete primero la Actividad 1 (variable pred).'
analisis = valid.copy()
analisis['pred'] = pred
analisis['error'] = (analisis['is_late'] != analisis['pred']).astype(int)
analisis['fp'] = ((analisis['pred'] == 1) & (analisis['is_late'] == 0)).astype(int)
analisis['fn'] = ((analisis['pred'] == 0) & (analisis['is_late'] == 1)).astype(int)

# TODO 3. Agrupe por mes_compra y customer_state.
def tabla_segmento(df, columna):
    t = df.groupby(columna).agg(
        pedidos=('order_id', 'count'),
        tardios_reales=('is_late', 'sum'),
        errores=('error', 'sum'),
        fp=('fp', 'sum'),
        fn=('fn', 'sum'),
        tasa_error_pct=('error', lambda s: 100 * s.mean()),
    )
    t['costo_R$'] = 8 * t['fp'] + 35 * t['fn']
    t['tasa_tardios_pct'] = 100 * t['tardios_reales'] / t['pedidos']
    return t

errores_mes = tabla_segmento(analisis, 'mes_compra').sort_index()
errores_estado = tabla_segmento(analisis, 'customer_state').sort_values(
    ['tasa_error_pct', 'pedidos'], ascending=[False, False])

print('Error por mes de compra:')
display(errores_mes.round(2))
print('Error por estado del cliente (todos, ordenado por tasa de error):')
display(errores_estado.round(2))
# Una tasa alta con pocos pedidos no es concluyente: se destacan los estados con >= 100 pedidos.
print('Estados con al menos 100 pedidos en validación:')
display(errores_estado[errores_estado['pedidos'] >= 100].round(2))

Error por mes de compra:


,pedidos,tardios_reales,errores,fp,fn,tasa_error_pct,costo_R$,tasa_tardios_pct
mes_compra,,,,,,,,
1,421,19,130,122,8,30.88,1256,4.51
2,422,61,142,117,25,33.65,1811,14.45
3,562,83,242,213,29,43.06,2719,14.77
4,530,26,159,153,6,30.00,1434,4.91
5,532,26,207,194,13,38.91,2007,4.89
6,526,11,118,112,6,22.43,1106,2.09
7,543,15,201,197,4,37.02,1716,2.76
8,603,37,264,255,9,43.78,2355,6.14
9,237,8,76,74,2,32.07,662,3.38


Error por estado del cliente (todos, ordenado por tasa de error):


,pedidos,tardios_reales,errores,fp,fn,tasa_error_pct,costo_R$,tasa_tardios_pct
customer_state,,,,,,,,
RR,3,0,3,3,0,100.00,24,0.00
SE,15,0,14,14,0,93.33,112,0.00
PA,45,1,35,35,0,77.78,280,2.22
PI,32,3,22,22,0,68.75,176,9.38
AL,25,3,17,17,0,68.00,136,12.00
MS,45,3,30,30,0,66.67,240,6.67
ES,124,7,77,76,1,62.10,643,5.65
RJ,682,99,423,405,18,62.02,3870,14.52
MA,34,4,21,21,0,61.76,168,11.76


Estados con al menos 100 pedidos en validación:


,pedidos,tardios_reales,errores,fp,fn,tasa_error_pct,costo_R$,tasa_tardios_pct
customer_state,,,,,,,,
ES,124,7,77,76,1,62.10,643,5.65
RJ,682,99,423,405,18,62.02,3870,14.52
BA,168,18,100,98,2,59.52,854,10.71
GO,101,3,52,51,1,51.49,443,2.97
SC,171,14,78,72,6,45.61,786,8.19
RS,313,25,135,122,13,43.13,1431,7.99
DF,122,7,37,33,4,30.33,404,5.74
PR,261,13,66,58,8,25.29,744,4.98
SP,2191,91,381,324,57,17.39,4587,4.15


### Actividad 4. Costo empresarial

Para el ejercicio, suponga un costo de R$ 8 por falso positivo y R$ 35 por falso negativo. Calcule el costo total de los errores en validación.

In [12]:
# TODO 4. Calcule el costo total con la matriz de confusión.
COSTO_FP, COSTO_FN = 8, 35   # supuestos didácticos de la guía (R$)
tn, fp, fn, tp = confusion_matrix(y_valid, pred, labels=[0, 1]).ravel()
costo_fp_fn = COSTO_FP * fp + COSTO_FN * fn

print(f'FP = {fp} -> R$ {COSTO_FP * fp:,.0f}')
print(f'FN = {fn} -> R$ {COSTO_FN * fn:,.0f}')
print(f'Costo total de errores (validación): R$ {costo_fp_fn:,.0f}')

# Referencia de costo: el modelo "siempre a tiempo" tiene todos los tardíos como FN.
costo_referencia = COSTO_FN * int((y_valid == 1).sum())
print(f'Costo del modelo de referencia (siempre a tiempo): R$ {costo_referencia:,.0f}')

# Complemento: costo según el umbral de decisión (misma validación; solo es ilustrativo).
umbrales = np.linspace(0.05, 0.95, 91)
filas = []
for u in umbrales:
    p = (proba >= u).astype(int)
    _, fp_, fn_, tp_ = confusion_matrix(y_valid, p, labels=[0, 1]).ravel()
    filas.append({'umbral': round(u, 2), 'fp': fp_, 'fn': fn_,
                  'recall': tp_ / (tp_ + fn_), 'costo': COSTO_FP * fp_ + COSTO_FN * fn_})
barrido = pd.DataFrame(filas)
mejor = barrido.loc[barrido['costo'].idxmin()]
print('\nUmbral con menor costo en validación:', mejor.to_dict())

FP = 1606 -> R$ 12,848
FN = 134 -> R$ 4,690
Costo total de errores (validación): R$ 17,538
Costo del modelo de referencia (siempre a tiempo): R$ 12,425

Umbral con menor costo en validación: {'umbral': 0.83, 'fp': 57.0, 'fn': 330.0, 'recall': 0.07042253521126761, 'costo': 12006.0}


### Autoverificación

Ejecute estas comprobaciones después de completar las actividades.

In [13]:
assert len(muestra) == 35000, 'La muestra debe tener 35000 pedidos.'
assert (len(train), len(valid), len(test)) == (24500, 5250, 5250), (
    'Particiones esperadas: 24500 / 5250 / 5250. Revise la celda de muestra.')
assert set(train['order_id']).isdisjoint(valid['order_id']), 'Train y valid comparten pedidos.'
assert set(train['order_id']).isdisjoint(test['order_id']), 'Train y test comparten pedidos.'
assert set(valid['order_id']).isdisjoint(test['order_id']), 'Valid y test comparten pedidos.'

assert modelo is not None and pred is not None, 'Complete la Actividad 1.'
assert len(pred) == len(y_valid), 'pred debe tener 5250 predicciones de validación.'
assert set(np.unique(np.asarray(pred))).issubset({0, 1}), 'pred debe contener solo clases 0 y 1.'
assert np.array_equal(np.asarray(pred), modelo.predict(X_valid)), (
    'pred no coincide con modelo.predict(X_valid). Verifique que utilizó X_valid.')

assert all(isinstance(v, (float, int, np.number)) for v in medidas.values()), (
    'Complete las métricas de la Actividad 2.')
assert proba is not None, 'Complete la Actividad 2: calcule las probabilidades.'
assert np.asarray(proba).ndim == 1 and len(proba) == len(y_valid), (
    'proba debe contener una probabilidad por pedido de validación.')
assert np.allclose(proba, modelo.predict_proba(X_valid)[:, 1]), (
    'proba no coincide con modelo.predict_proba(X_valid)[:, 1].')
assert np.isclose(medidas['f1'], f1_score(y_valid, pred, zero_division=0)), (
    "medidas['f1'] no coincide. Use f1_score(y_valid, pred, zero_division=0).")
assert np.isclose(medidas['ap'], average_precision_score(y_valid, proba)), (
    "medidas['ap'] no coincide. Use average_precision_score(y_valid, proba).")
assert np.isclose(medidas['roc_auc'], roc_auc_score(y_valid, proba)), (
    "medidas['roc_auc'] no coincide. Use roc_auc_score(y_valid, proba).")

assert isinstance(costo_fp_fn, (int, float, np.number)), (
    'Complete la Actividad 4: costo_fp_fn debe ser numérico (8*FP + 35*FN).')
_, fp_ver, fn_ver, _ = confusion_matrix(y_valid, pred).ravel()
assert np.isclose(costo_fp_fn, 8 * fp_ver + 35 * fn_ver), (
    'El costo no coincide con 8*FP + 35*FN. Revise la Actividad 4.')
print('Autoverificación superada.')


Autoverificación superada.


## 5. Ejemplo guiado de regresión

Con los mismos predictores, estime la duración total de la entrega en días usando `Ridge` y evalúe el error absoluto medio (MAE).

In [14]:
from sklearn.base import clone

# Cada modelo recibe una copia independiente del preprocesamiento.
modelo_reg = make_pipeline(clone(preprocesamiento), Ridge(solver='lsqr'))
modelo_reg.fit(X_train, train['dias_entrega'])
pred_dias = modelo_reg.predict(X_valid)
mae = mean_absolute_error(valid['dias_entrega'], pred_dias)
print('MAE regresión:', round(mae, 2), 'días')

MAE regresión: 5.07 días


1. Compare la regresión logística con el modelo de referencia utilizando Accuracy, Recall y F1. Justifique su elección de `class_weight`.

**Respuesta:**

Resultados en validación (5.250 pedidos, 355 tardíos = 6,76 %):

| Modelo | Accuracy | Recall | F1 |
|---|---|---|---|
| Referencia (siempre "a tiempo") | 0,932 | 0,000 | 0,000 |
| Reg. logística, `class_weight=None` | 0,933 | 0,003 | 0,006 |
| Reg. logística, `class_weight='balanced'` (elegida) | 0,669 | 0,623 | 0,203 |

La referencia tiene una Accuracy de 0,932 solo porque el 93 % de los pedidos llega a tiempo; no detecta ninguna entrega tardía. La regresión logística con `'balanced'` pierde unos 26 puntos de Accuracy, pero detecta 221 de los 355 pedidos tardíos (Recall 0,623) y el F1 sube de 0 a 0,203. Su Precision es baja (0,121): de cada 100 alertas, solo unas 12 son realmente tardías. El ROC-AUC (0,707) indica una capacidad moderada para ordenar los pedidos por riesgo, y el AP (0,160) es 2,4 veces la prevalencia (0,068): hay señal, pero limitada.

**Elección de `class_weight`:** elegí `'balanced'`. Con `None` el modelo solo emitió 1 alerta en 5.250 pedidos (Recall 0,003), es decir, se comporta igual que la referencia aunque su Accuracy parezca la mejor. Es interesante que el ROC-AUC y el AP son casi iguales en ambos casos (0,713 vs 0,707 y 0,160 vs 0,160): el ponderado no hace que el modelo "aprenda más", solo mueve el punto de corte efectivo para que se atreva a alertar. Con `'balanced'` el modelo es utilizable operativamente. Como contrapartida, con el umbral de 0,5 su costo (R$ 17.538) es mayor que el de `None` (R$ 12.390) y que el de la referencia (R$ 12.425), porque genera 1.606 falsos positivos; este punto se discute en la pregunta 2.


2. Analice qué tipo de pedidos presenta mayor error según los segmentos estudiados. ¿Cómo se relaciona esto con el costo de falsos positivos y falsos negativos?

**Respuesta:**

**Por mes de compra.** El error es mayor en agosto (43,8 %, 603 pedidos), marzo (43,1 %, 562), mayo (38,9 %) y julio (37,0 %), y menor en diciembre (13,2 %, 272), junio (22,4 %) y noviembre (25,0 %). Son segmentos de cientos de pedidos, por lo que la diferencia es razonablemente confiable. Los meses con error alto no coinciden con los de más retrasos reales: en agosto, julio y mayo la tasa real de tardíos es baja (6,1 %, 2,8 % y 4,9 %) y casi todo el error son falsos positivos (255 de 264 en agosto), mientras que febrero, marzo y noviembre (tasas reales de 14,5 %, 14,8 % y 12,8 %) concentran más falsos negativos (25, 29 y 20). Esto sugiere que el modelo no captura bien la estacionalidad de los retrasos.

**Por estado del cliente.** Las tasas más altas aparecen en RR (100 %), SE (93,3 %) y PA (77,8 %), pero con 3, 15 y 45 pedidos son poco concluyentes. Entre los estados con al menos 100 pedidos, los de mayor error son ES (62,1 %, 124 pedidos), RJ (62,0 %, 682), BA (59,5 %, 168) y GO (51,5 %, 101), y los de menor error son MG (17,3 %, 632) y SP (17,4 %, 2.191). Los estados del norte y nordeste y Río de Janeiro tienen más retrasos reales (RJ 14,5 %, BA 10,7 %) que SP (4,1 %), pero el modelo "balanced" los marca como riesgo de forma tan generalizada que erra en la mayoría de sus pedidos puntuales.

**Relación con el costo.** De los 1.740 errores, 1.606 (92 %) son falsos positivos y solo 134 son falsos negativos. Cada FP cuesta R$ 8 (total R$ 12.848) y cada FN R$ 35 (total R$ 4.690), para un costo de R$ 17.538. Aunque un FN es 4,4 veces más caro que un FP, el volumen de FP domina el costo. Por segmento, SP aporta el mayor costo (R$ 4.587) por su volumen (381 errores, 57 de ellos FN), seguido de RJ (R$ 3.870), donde coinciden un error alto y 99 tardíos reales. Un hallazgo importante es que la referencia que nunca alerta cuesta R$ 12.425, menos que el modelo con umbral 0,5. Al barrer umbrales sobre la validación, el menor costo (R$ 12.006) se logra con un umbral de 0,83, pero solo mejora a la referencia en 3,4 % y con un Recall de 0,07. Con estos costos, el modelo de la semana 12 no se justifica económicamente.

**Limitaciones:** los costos de R$ 8 y R$ 35 son supuestos didácticos, no costos reales de Olist; el umbral de 0,83 se eligió sobre la misma validación, por lo que su ahorro es optimista; y los segmentos pequeños (RR, SE, PA) no permiten concluir.


3. Proponga una mejora concreta para la semana 13. Considere la información del producto, del vendedor, del pago o el tratamiento de variables categóricas. Explique cómo verificará la mejora usando la misma partición de validación.

**Respuesta:**

**Propuesta.** El análisis de la pregunta 2 muestra que el error se concentra en rutas y épocas del año concretas, y que los estados del cliente y del vendedor entran al modelo lineal solo como indicadores independientes. Propongo dos cambios combinados:

1. **Variable de ruta:** crear `ruta = estado_vendedor → customer_state` (por ejemplo, SP→RJ) y la tasa histórica de tardíos de esa ruta calculada solo con `train` (con suavizado hacia la tasa global para rutas con pocos pedidos), junto con la relación entre `dias_prometidos` y esa ruta. Así el modelo puede detectar rutas en que el plazo prometido suele ser insuficiente. Se completa con la interacción `mes_compra` × región del cliente.
2. **Modelo no lineal con manejo nativo de categóricas** (por ejemplo `HistGradientBoostingClassifier`), que puede capturar interacciones entre ruta, mes, plazo y peso que la regresión logística no representa. Además, el umbral de decisión se elegirá minimizando `8*FP + 35*FN` mediante validación cruzada dentro de `train`, no sobre `valid`.

**Verificación.** Mantendré la misma muestra, las mismas particiones y el mismo preprocesamiento, ajustando todo (incluida la tasa por ruta) solo con `train` para evitar fuga de información. En `valid` compararé contra la línea base de esta semana: ROC-AUC 0,707, AP 0,160, Recall 0,623, F1 0,203 y costo R$ 17.538 (y la referencia de R$ 12.425). Consideraré que hay mejora si suben el ROC-AUC y el AP (métricas independientes del umbral) y si el costo con el umbral elegido en `train` queda por debajo de R$ 12.425. Además repetiré el análisis por mes y estado para comprobar que baja el error en ES, RJ y BA sin empeorar SP y MG.

**Limitaciones:** con 355 tardíos en validación las diferencias pequeñas pueden ser ruido (se revisará en la semana 14 con incertidumbre); la tasa por ruta puede sobreajustar en rutas con pocos pedidos; y los costos siguen siendo supuestos. `test` no se usará hasta la semana 16.


## 7. Registro de resultados

Guarde el resultado del laboratorio y la huella de la muestra. Las indicaciones de entrega están en el documento técnico.

In [15]:
import sys
import sklearn

reporte = {
    'laboratorio': 'S12',
    'id_estudiante': identificador,
    'semilla': semilla,
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sklearn.__version__,
    'n_muestra': len(muestra),
    'n_train': len(train),
    'n_valid': len(valid),
    'n_test': len(test),
    'n_variables': len(columnas),
    'columnas_predictoras': columnas,
    'fuente_olist_commit': 'fa0cc6d1aff713d3c8b9a1e633b56889881848bf',
    'tasa_tardias_muestra': float(muestra['is_late'].mean()),
    'huella_muestra': huella_muestra,
    'peso_clases': str(peso_clases),
    'metricas': {nombre: float(valor) for nombre, valor in medidas.items()},
    'costo_errores_brl': float(costo_fp_fn),
    'mae_regresion': float(mae),
}

ruta = Path(f'reports/S12_resultados_ID_{identificador}.json')
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding='utf-8')
print('Reporte:', ruta)

# En Colab, descarga el JSON para conservarlo fuera de la sesión.
try:
    from google.colab import files
    files.download(str(ruta))
except Exception:
    print('Si no se descargó automáticamente, guarde manualmente:', ruta)


Reporte: reports/S12_resultados_ID_01.json
Si no se descargó automáticamente, guarde manualmente: reports/S12_resultados_ID_01.json
